<a href="https://colab.research.google.com/github/Satyadeep-Dey/genai-lab/blob/main/11_1_RAG_Fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Introduction**

1.   This notebook explains the fundamentals of RAG without using LangChain or LlamaIndex
2.   I use an anonymized version of Tale of Two Cities as the knowledge base
3.   Chunking is done using a function I’ve written
4.   Embeddings are created using SentenceTransformer
5.   Embeddings are stored in FAISS
6.   LLM used is gpt-4.1-mini


**To ensure pure grounded RAG, the following is done -**


1.   Similarity threshold - This is the most important safeguard.
2.   Strict prompt instruction - Even if retrieval returns weak matches, the LLM must refuse to invent answers.
3.   Show source chunks - This makes system auditable.




In [ ]:
#Install required packages
!pip install -q openai faiss-cpu

In [ ]:
#Imports
import sys
from google.colab import drive
import faiss
from openai import OpenAI
from google.colab import userdata

In [ ]:
#Setup an LLM that can answer questions based on Context

#define constants
MODEL = "gpt-4.1-mini"
#MODEL = "gpt-5.2"

# Sign in to OpenAI using Secrets in Colab
openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)

# Check if Open AI key has been set
if openai_api_key:
    print("OpenAI API Key exists")
else:
    print("OpenAI API Key not set")

In [ ]:
#Function to call ChatGPT

def message_gpt(system_message,prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    completion = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
    )
    return completion.choices[0].message.content

In [ ]:
# import a re-use function to read text from file stored in Google Drive
drive.mount('/content/drive', force_remount=True)
UTIL_PATH = "/content/drive/MyDrive/Colab Notebooks"
sys.path.insert(0, UTIL_PATH)
from file_utils_colab import read_text_from_file


In [ ]:
# Let's use an anonymized and abridged version of Tale of Two Cities for RAG knowledge base

KB_text = read_text_from_file(
    folder_path="Files/Knowledge-Base",
    file_name= "Anonymized by OpenAI_TOTC.txt"
)

In [ ]:
print(f"The number of characters are : {len(KB_text)}")

number_of_words = len(KB_text.split())
# Divides a string into a list of substrings based on a specified separator (default is whitespace) and then counts length of list

print(f"Number of words is : {number_of_words}")
#print(KB_text)

In [ ]:
# Step 1 : Chunking the document
def chunk_text(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

In [ ]:
chunks = chunk_text(KB_text) #,chunk_size=50, overlap=10)
print(f"Total number of chunks: {len(chunks)}")
print("-----------------------------------------------------------")
print(f"Chunk 1 is : {chunks[0]}")
print("-----------------------------------------------------------")
print(f"Chunk 2 is : {chunks[1]}") # just to see the overlap

In [ ]:
# Step 2: Create embeddings using Hugging Face embedding model

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks)

In [ ]:
# Tell LLM to answwer only with Context
# Strict prompt instruction

system_message = """You are a helpful assistant who will answer questions only based in the context that is shared with you.
If the question is about something that is not in the context given to you then you'll respond with DO-NOT-KNOW .
Do not lie or make things up.
"""
context = ""

In [ ]:
#Step 3: Store vectors generated by embedding in FAISS
# ..which unlike Chroma DB stores only vectors

# Normalize embeddings before indexing so that similarity will be between -1 and 1 .
# Needed for checking Similarity threshold

faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

In [ ]:
# Step 4 :
# Retrieve chunks that are relevant to the question
# Check if they are similar enough using a threshold
# IMPORTANT TO USE THRESHOLD TO IMPROVE ANSWER QUALITY
# then pass as context to the LLM along with question
# so that LLM can answer based on this context

threshold = 0.5
question = "Who is the resurrection man?"

q_embedding = model.encode([question])

faiss.normalize_L2(q_embedding)#Normalize query too

D, I = index.search(q_embedding, k=5)
# k = number of chunks
# D = similarity scores (or distances)
# I = indices of similar chunks.
retrieved_chunks = [chunks[i] for i in I[0]]
print(f"Retrieved chunks is : {retrieved_chunks}")

# let's see similarity score for each chunk
print("----------------------------------------")
for score, idx in zip(D[0], I[0]):
    print(f"Score: {score}, Chunk: {chunks[idx]}") #print score and data in that chunk
    print("----------------------------------------")

# Check for threshold to prevent hallucination
filtered_chunks = []
for score, idx in zip(D[0], I[0]):
    if score >= threshold:
        filtered_chunks.append(chunks[idx])

if not filtered_chunks:
    print("No relevant chunks found.")
else:
    context = "\n".join(filtered_chunks)
    print("Context passed to LLM:\n", context)

In [ ]:
# Step 5: Prompt Construction
# See the strict prompt instruction
prompt = "This is the context based on which you should answer : " + context + ". And this is the question : " + question

In [ ]:
# Step 6: Generate Answer
message_gpt(system_message,prompt)

In [ ]:
# Let's combine into a single function
def get_answer(question,threshold = 0.45,debug  = False):

    q_embedding = model.encode([question])
    faiss.normalize_L2(q_embedding)

    D, I = index.search(q_embedding, k=5)

    filtered_chunks = []
    for score, idx in zip(D[0], I[0]):
      if score >= threshold:
        filtered_chunks.append(chunks[idx])

    if debug == True:
      # let's see similarity score for each chunk
      print("----------------------------------------")
      for score, idx in zip(D[0], I[0]):
        print(f"Score: {score}, Chunk: {chunks[idx]}") #print score and data in that chunk
        print("----------------------------------------")

    if not filtered_chunks:
      return {"answer": "DO-NOT-KNOW", "sources": "NA"}
    else:
      context = "\n".join(filtered_chunks) # add this to context
      prompt = "This is the context based on which you should answer : " + context + ". And this is the question : " + question
      answer = message_gpt(system_message,prompt)
      final_output = {"answer": answer, "sources": filtered_chunks}
      return final_output # enter answer in a JSON format so that another program can take it as input

In [ ]:
print(get_answer("Who is the author of A Chronicle of Two Cities"))

In [ ]:
print(get_answer("Who is Philippe Duval married to ?"))

In [ ]:
print(get_answer("Who are the owners of the wine shop in this story ?"))

In [ ]:
print(get_answer("Who is on trial for treason against England ?"))

In [ ]:
print(get_answer("Who is the Jackal and who is the Lion ?"))

In [ ]:
print(get_answer("Why is he on trial ?")) #retains context from previous question

In [ ]:
print(get_answer("Who are the witnesses in this trial ?")) #retains context from previous question

In [ ]:
print(get_answer("What is the capital of India ?")) # can't answer since it's not in knowledge base

In [ ]:
print(get_answer("Which cities are named in this story ?"))

In [ ]:
print(get_answer("Who is the villain in this story ?",0.35))
# threshold always does a similarity search between question and KB .But in this case similarity score is very low
# setting threshold to lower value that chunks are sent to LLM based on which it can answer this indirect question